# Exercícios — TextFlix

1) **Parser do catálogo**  
Implemente a função que recebe a *string* inicial e retorna a lista de dicionários no formato especificado (`id`, `titulo`, `ano`, `generos`, `sinopse`, `visto`). **Itens inválidos devem ser descartados**.

2) **Cadastrar novo filme**  
Implemente uma função que recebe do usuário **título, ano, gêneros e sinopse**, criando um novo dicionário no formato do catálogo e **adicionando-o** à lista.

3) **Remover filme**  
Crie uma função que permita ao usuário **informar o título** de um filme e **removê-lo** da lista, se existir.

4) **Listar itens**  
Faça uma função que **mostre todos os filmes** do catálogo, permitindo **ordenar** por **id** (padrão), **título** ou **ano** de acordo com o input do usuário.

5) **Buscar no título (AND/OR)**  
Implemente uma busca em que o usuário digita **palavras** e pode escolher se a busca será do tipo **AND** (todas as palavras devem aparecer) ou **OR** (qualquer palavra encontrada já conta).

6) **Buscar por substring**  
Crie uma função que **retorne todos os filmes** cujo **título contenha** uma determinada **substring** informada pelo usuário, **exibindo-os ordenados pelo ano**.

7) **Filtrar por gênero**  
Implemente uma função que retorne todos os filmes que **contenham um gênero específico** especificado pelo usuário. **Ordene** a saída pelo **título** dos filmes.

8) **Marcar como visto**  
Faça uma função que permita ao usuário **indicar que assistiu** a um filme (informando o título), alterando o campo **`visto` para `True`**.

9) **Listar não vistos**  
Crie uma função que percorra o catálogo e **mostre apenas os filmes ainda não vistos** pelo usuário.

10) **Abreviar sinopse**  
Implemente uma função que **imprima a sinopse abreviada** de cada filme.  
- Se a sinopse tiver **até 50 caracteres**, mostre **normalmente**.  
- Se tiver **mais de 30**, percorra **palavra por palavra** e **abrevie** as que possuam **mais de 6 letras**, mantendo as **seis primeiras** e acrescentando um **ponto final**.  
- Se a **sexta letra** for uma **vogal**, **retire letras** até terminar em **consoante** antes de acrescentar o ponto.  
Ex.: `"computador potente para simulação"` → `"comput. potent. para simul."`.

11) **Top-k palavras**  
Crie uma função que mostre as **k palavras mais frequentes** do catálogo (considerando **títulos e sinopses** juntos). **Apenas palavras com pelo menos 5 caracteres** devem entrar no cálculo.

12) **Ranking de relevância**  
Implemente uma função que recebe **palavras-chave** (separadas por vírgula) e um número **k**.  
Calcule, para cada filme, um **score** = número de ocorrências dessas palavras na **sinopse** (**apenas palavras com 5+ caracteres**).  
Liste os **k filmes mais relevantes** em **ordem decrescente** de score. Se **k** for maior que o número de filmes, **liste todos**.


In [ ]:

# ----------------------------
# Dados em memória
# ----------------------------
catalogo = []
_proximo_id = 1

def _next_id():
    """Gera id incremental para novos itens."""
    global _proximo_id
    nid = _proximo_id
    _proximo_id += 1
    return nid

# ----------------------------
# Utilitários de texto
# ----------------------------
def normalizar_generos(s):
    """Converte 'drama, Ação ; sci-fi' -> ['drama','ação','sci-fi'] (minúsculas, sem vazios, sem duplicatas)."""
    if s is None:
        return []
    texto = str(s).replace(";", ",")
    partes = [p.strip().lower() for p in texto.split(",")]
    out = []
    for g in partes:
        if g and g not in out:
            out.append(g)
    return out

def _tokenizar_basico(texto):
    """Tokeniza removendo pontuação simples: mantém letras/dígitos e separa por espaço (sem regex)."""
    t = str(texto)
    for ch in ",.;:!?()[]{}\"'`“”’–—-_*/\\|@#%$^&+=<>~\t\n\r\v\f":
        t = t.replace(ch, " ")
    return " ".join(t.split()).split(" ") if t.strip() else []

def _strip_punct(tok):
    """Separa prefixo/sufixo de pontuação simples e miolo; útil para mascarar palavras com pontuação colada."""
    punct = ".,;:!?()[]{}\"'“”’–—-_"
    i, j = 0, len(tok) - 1
    while i <= j and tok[i] in punct:
        i += 1
    while j >= i and tok[j] in punct:
        j -= 1
    return tok[:i], tok[i:j+1], tok[j+1:]

# ----------------------------
# Funcionalidades
# ----------------------------
def cadastrar_item(titulo, ano, generos_str, sinopse):
    """Cadastra um item no catálogo e retorna o dicionário criado."""
    item = {
        "id": _next_id(),
        "titulo": " ".join(str(titulo).split()),
        "ano": int(ano),
        "generos": normalizar_generos(generos_str),
        "sinopse": str(sinopse),
    }
    catalogo.append(item)
    return item

def listar_itens(ordenar_por='id'):
    """Retorna a lista ordenada segundo o critério informado: id, titulo (A→Z) ou ano_desc."""
    if ordenar_por == "titulo":
        return sorted(catalogo, key=lambda it: it.get("titulo","").lower())
    if ordenar_por == "ano_desc":
        return sorted(catalogo, key=lambda it: it.get("ano", 0), reverse=True)
    return sorted(catalogo, key=lambda it: it.get("id", 0))

def buscar_titulo(palavras, modo='AND'):
    """Busca por palavras no título, modo AND/OR (case-insensitive)."""
    if isinstance(palavras, str):
        ws = [w for w in palavras.split() if w]
    else:
        ws = [str(w) for w in (palavras or []) if str(w)]
    ws = [w.lower() for w in ws]
    if not ws:
        return []
    if modo.upper() == "OR":
        cond = lambda it: any(w in it.get("titulo","").lower() for w in ws)
    else:
        cond = lambda it: all(w in it.get("titulo","").lower() for w in ws)
    return list(filter(cond, catalogo))

def filtrar_generos(generos_req):
    """Retorna itens que contenham todos os gêneros solicitados (case-insensitive)."""
    req = [g.lower() for g in (generos_req or []) if str(g).strip()]
    if not req:
        return []
    return list(filter(lambda it: all(g in it.get("generos", []) for g in req), catalogo))

def acronimo_titulo(id_item, stop=None):
    """Acrônimo do título (maiúsculas), ignorando stopwords (case-insensitive)."""
    stop_lower = {w.lower() for w in (stop or [])}
    for it in catalogo:
        if it.get("id") == id_item:
            palavras = it.get("titulo","").split()
            kept = filter(lambda w: w.lower() not in stop_lower, palavras)
            letras = list(map(lambda w: w[0].upper(), filter(lambda w: len(w)>0, kept)))
            return "".join(letras)
    return ""

def slug_titulo(id_item):
    """Gera slug: minúsculas, palavras separadas por '-', sem pontuação extra."""
    for it in catalogo:
        if it.get("id") == id_item:
            t = it.get("titulo","").lower()
            for ch in ",.;:!?()[]{}\"'“”’–—-_*/\\|@#%$^&+=<>~\t\n\r\v\f":
                t = t.replace(ch, " ")
            return "-".join(t.split())
    return ""

def limpar_sinopse(id_item, proibidas):
    """Mascara palavras proibidas na sinopse (case-insensitive), preservando pontuação colada. Colapsa espaços no fim."""
    proib = {str(w).lower() for w in (proibidas or []) if str(w).strip()}
    for it in catalogo:
        if it.get("id") == id_item:
            palavras = it.get("sinopse","").split()
            out = []
            for tok in palavras:
                pre, core, suf = _strip_punct(tok)
                if core.lower() in proib and core != "":
                    core = "***"
                out.append(pre + core + suf)
            it["sinopse"] = " ".join(" ".join(out).split())
            return True
    return False

def top_palavras(k, min_len=3, stop=None):
    """Top-k palavras do catálogo (títulos + sinopses), ordenadas por cont desc e, em empate, por palavra asc."""
    stop_lower = {w.lower() for w in (stop or [])}
    todas = []
    for it in catalogo:
        todas.extend(_tokenizar_basico(it.get("titulo","")))
        todas.extend(_tokenizar_basico(it.get("sinopse","")))
    toks = filter(lambda w: len(w) >= int(min_len), todas)
    toks = filter(lambda w: w.lower() not in stop_lower, toks)
    cont = {}
    for w in toks:
        wl = w.lower()
        cont[wl] = cont.get(wl, 0) + 1
    pares = list(cont.items())
    return sorted(pares, key=lambda t: (-t[1], t[0]))[:int(k)]

def sugerir_tags(id_item, min_len=4):
    """Extrai palavras boas de título/sinopse (minúsculas, sem duplicatas, ordem preservada)."""
    for it in catalogo:
        if it.get("id") == id_item:
            toks = _tokenizar_basico(it.get("titulo","")) + _tokenizar_basico(it.get("sinopse",""))
            out = []
            for w in map(lambda s: s.lower(), toks):
                if len(w) >= int(min_len) and w not in out:
                    out.append(w)
            return out
    return []

def ids_duplicados():
    """Retorna ids de itens com títulos duplicados (case-insensitive), preservando o primeiro como original."""
    vistos = []
    dups = []
    for it in catalogo:
        k = it.get("titulo","").lower()
        if k in vistos:
            dups.append(it.get("id"))
        else:
            vistos.append(k)
    return dups

def top_densidade(n):
    """Top-n por (len(titulo) + len(sinopse)), desempate por título (asc, case-insensitive)."""
    pares = map(lambda it: (it.get("id"), it.get("titulo",""), len(it.get("titulo","")) + len(it.get("sinopse",""))), catalogo)
    return sorted(pares, key=lambda t: (-t[2], t[1].lower()))[:int(n)]

def itens_por_inicial():
    """Conta itens por inicial do título (A–Z, demais -> '#'), ordenado pela inicial."""
    cont = {}
    for it in catalogo:
        t = it.get("titulo","").strip()
        ini = t[:1].upper() if t[:1] else "#"
        if not ("A" <= ini <= "Z"):
            ini = "#"
        cont[ini] = cont.get(ini, 0) + 1
    pares = list(cont.items())
    return sorted(pares, key=lambda t: t[0])

# ----------------------------
# Menu CLI
# ----------------------------
def menu():
    while True:
        print("\n=== TextoFlix CLI ===")
        print("1) Cadastrar item")
        print("2) Listar itens")
        print("3) Buscar no título (AND/OR)")
        print("4) Filtrar por gêneros (todos)")
        print("5) Acrônimo do título")
        print("6) Slug do título")
        print("7) Limpar sinopse (palavras proibidas)")
        print("8) Top palavras do catálogo")
        print("9) Sugerir tags de um item")
        print("10) IDs com títulos duplicados")
        print("11) Top densidade de texto")
        print("12) Relatório por iniciais")
        print("0) Sair")
        op = input("Opção: ").strip()

        if op == "1":
            titulo = input("Título: ")
            ano = input("Ano: ")
            generos = input("Gêneros (separados por vírgula/;): ")
            sinopse = input("Sinopse: ")
            try:
                it = cadastrar_item(titulo, int(ano), generos, sinopse)
                print(f"OK. Item cadastrado com id {it['id']}.")
            except Exception as e:
                print("Erro ao cadastrar:", e)

        elif op == "2":
            crit = input("Ordenar por [id|titulo|ano_desc]: ").strip().lower() or "id"
            itens = listar_itens(crit)
            for it in itens:
                print(f"{it['id']:>3} | {it['titulo']} ({it['ano']})  gêneros={','.join(it['generos'])}\n    {it['sinopse']}")

        elif op == "3":
            termo = input("Palavras (separe por espaço): ").strip()
            modo = input("Modo [AND/OR]: ").strip().upper() or "AND"
            resultados = buscar_titulo(termo, modo)
            for it in resultados:
                print(f"{it['id']:>3} | {it['titulo']} ({it['ano']})")

        elif op == "4":
            gs = input("Gêneros requeridos (vírgula/;): ").strip()
            req = normalizar_generos(gs)
            resultados = filtrar_generos(req)
            for it in resultados:
                print(f"{it['id']:>3} | {it['titulo']} ({it['ano']})  gêneros={','.join(it['generos'])}")

        elif op == "5":
            try:
                i = int(input("ID do item: ").strip())
            except:
                print("ID inválido."); continue
            stop = input("Stopwords (opcional, separadas por espaço): ").split()
            print("Acrônimo:", acronimo_titulo(i, stop=stop))

        elif op == "6":
            try:
                i = int(input("ID do item: ").strip())
            except:
                print("ID inválido."); continue
            print("Slug:", slug_titulo(i))

        elif op == "7":
            try:
                i = int(input("ID do item: ").strip())
            except:
                print("ID inválido."); continue
            proib = input("Palavras proibidas (espaço): ").split()
            print("Limpou?" , limpar_sinopse(i, proib))

        elif op == "8":
            try:
                k = int(input("Top-k: ").strip())
            except:
                print("k inválido."); continue
            stop = input("Stopwords (opcional, espaço): ").split()
            for (w,c) in top_palavras(k, stop=stop):
                print(f"{w:15} : {c}")

        elif op == "9":
            try:
                i = int(input("ID do item: ").strip())
            except:
                print("ID inválido."); continue
            minl = int(input("Tamanho mínimo (default 4): ").strip() or "4")
            print(sugerir_tags(i, min_len=minl))

        elif op == "10":
            print(ids_duplicados())

        elif op == "11":
            try:
                n = int(input("Top-n: ").strip())
            except:
                print("n inválido."); continue
            for (idv, tit, score) in top_densidade(n):
                print(f"{idv:>3} | {tit}  score={score}")

        elif op == "12":
            for (ini, qtd) in itens_por_inicial():
                print(f"{ini}: {qtd}")

        elif op == "0":
            print("Até mais!"); break
        else:
            print("Opção inválida.")

# ----------------------------
# Sementes de dados (opcional) e execução
# ----------------------------
def _seed():
    cadastrar_item("Matrix", 1999, "ação, sci-fi", "Um hacker descobre a verdade sobre sua realidade.")
    cadastrar_item("Cidade de Deus", 2002, "drama", "História de ascensão do crime organizado nas favelas do Rio.")
    cadastrar_item("O Senhor dos Anéis: A Sociedade do Anel", 2001, "aventura; fantasia", "Jornada épica pela Terra-média.")

if __name__ == "__main__":
    # Descomente para testar rapidamente:
    # _seed()
    # menu()
    pass
